# VeriHire ML Service - Google Colab

This notebook runs the ML models for VeriHire:
- **CodeBERT**: Code quality evaluation
- **BERT**: Written response analysis
- **NCF**: Candidate-job matching

## Setup Instructions
1. Go to **Runtime > Change runtime type** and select **T4 GPU**
2. Run all cells in order
3. Copy the ngrok URL at the end and paste it in your `.env` file as `ML_SERVICE_URL`

---

## 1. Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok transformers torch numpy scikit-learn nest-asyncio pydantic

## 2. Set up ngrok (for public URL)

Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# @title Enter your ngrok auth token { display-mode: "form" }
NGROK_AUTH_TOKEN = "" # @param {type:"string"}

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok auth token set!")
else:
    print("WARNING: No ngrok token set. Get one free at https://dashboard.ngrok.com/get-started/your-authtoken")

## 3. Define the ML Service Code

In [ ]:
import torch
import numpy as np
import re
import time
import logging
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Optional
from collections import Counter

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============ Pydantic Schemas ============

class ProgrammingLanguage(str, Enum):
    PYTHON = "python"
    JAVASCRIPT = "javascript"
    TYPESCRIPT = "typescript"
    JAVA = "java"
    CSHARP = "csharp"
    CPP = "cpp"
    GO = "go"
    RUST = "rust"

class TextEvaluationType(str, Enum):
    WRITTEN_RESPONSE = "written_response"
    PEER_REVIEW = "peer_review"
    DESIGN_EXPLANATION = "design_explanation"

# Code Evaluation
class CodeEvaluationRequest(BaseModel):
    code: str
    language: ProgrammingLanguage
    challenge_description: Optional[str] = None

class CodeMetrics(BaseModel):
    complexity_score: float
    readability_score: float
    maintainability_score: float
    security_score: float
    best_practices_score: float

class CodeIssue(BaseModel):
    severity: str
    category: str
    message: str
    line: Optional[int] = None

class CodeEvaluationResponse(BaseModel):
    overall_score: float
    metrics: CodeMetrics
    issues: list[CodeIssue] = []
    strengths: list[str] = []
    suggestions: list[str] = []
    processing_time_ms: float

# Text Evaluation
class TextEvaluationRequest(BaseModel):
    text: str
    evaluation_type: TextEvaluationType = TextEvaluationType.WRITTEN_RESPONSE
    question: Optional[str] = None
    expected_topics: Optional[list[str]] = None

class TextMetrics(BaseModel):
    relevance_score: float
    coherence_score: float
    depth_score: float
    clarity_score: float
    originality_score: float

class TextEvaluationResponse(BaseModel):
    overall_score: float
    metrics: TextMetrics
    topics_covered: list[str] = []
    key_points: list[str] = []
    suggestions: list[str] = []
    word_count: int
    processing_time_ms: float

# Candidate Matching
class SkillRequirement(BaseModel):
    skill_id: str
    skill_name: str
    required_level: int = Field(ge=1, le=5)
    weight: float = 1.0
    is_required: bool = True

class CandidateSkill(BaseModel):
    skill_id: str
    skill_name: str
    proficiency_level: int = Field(ge=1, le=5)
    verified: bool = False
    certification_score: Optional[float] = None

class CandidateProfile(BaseModel):
    candidate_id: str
    skills: list[CandidateSkill]
    experience_years: Optional[float] = None

class JobProfile(BaseModel):
    job_id: str
    required_skills: list[SkillRequirement]
    experience_min: Optional[float] = None
    experience_max: Optional[float] = None

class MatchScore(BaseModel):
    candidate_id: str
    job_id: str
    overall_score: float
    skill_match_score: float
    experience_match_score: float
    ncf_score: float
    skill_gaps: list[str] = []
    skill_strengths: list[str] = []

class CandidateMatchRequest(BaseModel):
    job: JobProfile
    candidates: list[CandidateProfile]
    top_k: int = 10
    min_score: float = 0.0

class CandidateMatchResponse(BaseModel):
    job_id: str
    matches: list[MatchScore]
    total_candidates: int
    processing_time_ms: float

# Health
class ModelStatus(BaseModel):
    name: str
    loaded: bool
    device: str

class HealthResponse(BaseModel):
    status: str
    version: str
    timestamp: str
    models: Optional[list[ModelStatus]] = None

print("Schemas defined!")

In [ ]:
# ============ CodeBERT Service ============

class CodeBERTService:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = device
        self.loaded = False

    def load_model(self):
        if self.loaded:
            return
        logger.info("Loading CodeBERT model...")
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        self.model = AutoModel.from_pretrained("microsoft/codebert-base")
        self.model.to(self.device)
        self.model.eval()
        self.loaded = True
        logger.info(f"CodeBERT loaded on {self.device}")

    def get_embedding(self, code: str) -> torch.Tensor:
        inputs = self.tokenizer(code, return_tensors="pt", truncation=True, max_length=512, padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.last_hidden_state[:, 0, :]

    def analyze_complexity(self, code: str) -> float:
        lines = code.strip().split("\n")
        max_depth = 0
        for line in lines:
            stripped = line.lstrip()
            if stripped:
                indent = len(line) - len(stripped)
                depth = indent // 4
                max_depth = max(max_depth, depth)

        control_patterns = [r"\bif\b", r"\bfor\b", r"\bwhile\b", r"\btry\b", r"\bswitch\b"]
        control_count = sum(len(re.findall(p, code, re.IGNORECASE)) for p in control_patterns)

        depth_penalty = min(max_depth / 10, 0.3)
        control_penalty = min(control_count / 20, 0.3)
        return round(max(0.0, 1.0 - depth_penalty - control_penalty), 3)

    def analyze_readability(self, code: str) -> float:
        lines = [l for l in code.strip().split("\n") if l.strip()]
        if not lines:
            return 0.0

        avg_line_length = sum(len(l) for l in lines) / len(lines)
        line_score = max(0.0, 1.0 - (avg_line_length - 40) / 80) if avg_line_length > 40 else 1.0

        comments = len(re.findall(r"#.*$|//.*$|/\*[\s\S]*?\*/", code, re.MULTILINE))
        comment_score = min(comments / len(lines) * 5, 1.0)

        return round(line_score * 0.5 + comment_score * 0.5, 3)

    def analyze_security(self, code: str) -> tuple[float, list[CodeIssue]]:
        issues = []
        patterns = [
            (r"\beval\s*\(", "Use of eval() is dangerous", "security"),
            (r"\bexec\s*\(", "Use of exec() is dangerous", "security"),
            (r"password\s*=\s*['\"][^'\"]+['\"]", "Hardcoded password", "security"),
            (r"api[_-]?key\s*=\s*['\"][^'\"]+['\"]", "Hardcoded API key", "security"),
        ]
        for pattern, message, category in patterns:
            matches = list(re.finditer(pattern, code, re.IGNORECASE))
            for match in matches:
                line_num = code[:match.start()].count("\n") + 1
                issues.append(CodeIssue(severity="warning", category=category, message=message, line=line_num))

        score = max(0.0, 1.0 - len(issues) * 0.15)
        return round(score, 3), issues

    def analyze_best_practices(self, code: str) -> tuple[float, list[CodeIssue]]:
        issues = []
        patterns = [
            (r"except\s*:", "Bare except clause", "warning"),
            (r"TODO|FIXME", "Unresolved TODO/FIXME", "info"),
            (r"print\s*\(", "Debug print statement", "info"),
            (r"console\.log\s*\(", "Debug console.log", "info"),
        ]
        for pattern, message, severity in patterns:
            matches = list(re.finditer(pattern, code, re.MULTILINE))
            for match in matches:
                line_num = code[:match.start()].count("\n") + 1
                issues.append(CodeIssue(severity=severity, category="best_practice", message=message, line=line_num))

        penalty = len([i for i in issues if i.severity == "warning"]) * 0.1 + len([i for i in issues if i.severity == "info"]) * 0.03
        return round(max(0.0, 1.0 - penalty), 3), issues

    def evaluate(self, request: CodeEvaluationRequest) -> CodeEvaluationResponse:
        start = time.time()

        complexity = self.analyze_complexity(request.code)
        readability = self.analyze_readability(request.code)
        maintainability = (complexity + readability) / 2
        security, sec_issues = self.analyze_security(request.code)
        best_practices, bp_issues = self.analyze_best_practices(request.code)

        all_issues = sec_issues + bp_issues

        metrics = CodeMetrics(
            complexity_score=complexity,
            readability_score=readability,
            maintainability_score=maintainability,
            security_score=security,
            best_practices_score=best_practices,
        )

        overall = (complexity * 0.15 + readability * 0.20 + maintainability * 0.20 + security * 0.25 + best_practices * 0.20) * 100

        strengths = []
        if complexity >= 0.8: strengths.append("Low code complexity")
        if readability >= 0.8: strengths.append("Good readability")
        if security >= 0.9: strengths.append("No security issues detected")

        suggestions = []
        if complexity < 0.6: suggestions.append("Consider breaking down complex functions")
        if readability < 0.6: suggestions.append("Add more comments and use descriptive names")
        if security < 0.8: suggestions.append("Review security vulnerabilities")

        return CodeEvaluationResponse(
            overall_score=round(overall, 2),
            metrics=metrics,
            issues=all_issues,
            strengths=strengths,
            suggestions=suggestions,
            processing_time_ms=round((time.time() - start) * 1000, 2),
        )

codebert_service = CodeBERTService()
print("CodeBERT service defined!")

In [ ]:
# ============ BERT Service ============

class BERTService:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = device
        self.loaded = False

    def load_model(self):
        if self.loaded:
            return
        logger.info("Loading BERT model...")
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        self.model = AutoModel.from_pretrained("bert-base-uncased")
        self.model.to(self.device)
        self.model.eval()
        self.loaded = True
        logger.info(f"BERT loaded on {self.device}")

    def analyze_relevance(self, text: str, question: Optional[str], topics: Optional[list[str]]) -> float:
        if not question and not topics:
            return 0.7

        text_lower = text.lower()
        text_words = set(re.findall(r"\b\w+\b", text_lower))
        score = 0.0
        checks = 0

        if question:
            q_words = set(re.findall(r"\b\w+\b", question.lower()))
            stop_words = {"the", "a", "an", "is", "are", "what", "how", "why", "to", "for", "of", "in"}
            q_words = q_words - stop_words
            if q_words:
                overlap = len(text_words & q_words) / len(q_words)
                score += min(overlap * 1.5, 1.0)
                checks += 1

        if topics:
            found = sum(1 for t in topics if any(w in text_lower for w in t.lower().split()))
            score += found / len(topics)
            checks += 1

        return round(score / max(checks, 1), 3)

    def analyze_coherence(self, text: str) -> float:
        sentences = re.split(r"[.!?]+", text)
        sentences = [s.strip() for s in sentences if s.strip()]
        if len(sentences) <= 1:
            return 0.5

        transitions = [r"\bhowever\b", r"\btherefore\b", r"\bfurthermore\b", r"\bfor example\b", r"\bin conclusion\b"]
        trans_count = sum(1 for p in transitions if re.search(p, text, re.IGNORECASE))
        return round(min(trans_count / (len(sentences) * 0.3), 1.0) * 0.7 + 0.3, 3)

    def analyze_depth(self, text: str) -> float:
        word_count = len(re.findall(r"\b\w+\b", text))
        length_score = min(word_count / 150, 1.0)

        technical = [r"\d+%", r"algorithm", r"complexity", r"API", r"database", r"function", r"method"]
        tech_count = sum(len(re.findall(p, text, re.IGNORECASE)) for p in technical)
        tech_score = min(tech_count / 5, 1.0)

        return round(length_score * 0.6 + tech_score * 0.4, 3)

    def analyze_clarity(self, text: str) -> float:
        sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
        if not sentences:
            return 0.0

        lengths = [len(s.split()) for s in sentences]
        avg = sum(lengths) / len(lengths)
        if 10 <= avg <= 25:
            return 1.0
        elif avg < 10:
            return avg / 10
        else:
            return max(0.3, 1.0 - (avg - 25) / 30)

    def analyze_originality(self, text: str) -> float:
        words = re.findall(r"\b\w+\b", text.lower())
        if not words:
            return 0.0
        unique = set(words)
        return round(min(len(unique) / len(words) * 2, 1.0), 3)

    def extract_topics(self, text: str) -> list[str]:
        words = re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())
        stop = {"that", "this", "with", "from", "have", "been", "were", "they", "will", "would", "about"}
        filtered = [w for w in words if w not in stop]
        counts = Counter(filtered)
        return [w for w, c in counts.most_common(5) if c >= 2]

    def evaluate(self, request: TextEvaluationRequest) -> TextEvaluationResponse:
        start = time.time()

        word_count = len(re.findall(r"\b\w+\b", request.text))

        relevance = self.analyze_relevance(request.text, request.question, request.expected_topics)
        coherence = self.analyze_coherence(request.text)
        depth = self.analyze_depth(request.text)
        clarity = self.analyze_clarity(request.text)
        originality = self.analyze_originality(request.text)

        metrics = TextMetrics(
            relevance_score=relevance,
            coherence_score=coherence,
            depth_score=depth,
            clarity_score=clarity,
            originality_score=originality,
        )

        overall = (relevance * 0.25 + coherence * 0.20 + depth * 0.25 + clarity * 0.15 + originality * 0.15) * 100

        topics = self.extract_topics(request.text)

        suggestions = []
        if relevance < 0.6: suggestions.append("Focus more on the question/topic")
        if depth < 0.6: suggestions.append("Provide more specific examples")
        if clarity < 0.6: suggestions.append("Simplify sentence structure")

        return TextEvaluationResponse(
            overall_score=round(overall, 2),
            metrics=metrics,
            topics_covered=topics,
            key_points=[],
            suggestions=suggestions,
            word_count=word_count,
            processing_time_ms=round((time.time() - start) * 1000, 2),
        )

bert_service = BERTService()
print("BERT service defined!")

In [ ]:
# ============ NCF Service ============

class NCFService:
    def __init__(self):
        self.device = device
        self.loaded = True  # NCF uses rule-based + simple ML, always ready

    def load_model(self):
        pass  # NCF is lightweight, no heavy model to load

    def calculate_skill_match(self, candidate: CandidateProfile, job: JobProfile) -> tuple[float, list[str], list[str]]:
        if not job.required_skills:
            return 1.0, [], []

        candidate_lookup = {s.skill_name.lower(): s for s in candidate.skills}
        total_weight = sum(r.weight for r in job.required_skills) or len(job.required_skills)

        matched_score = 0.0
        gaps = []
        strengths = []

        for req in job.required_skills:
            cskill = candidate_lookup.get(req.skill_name.lower())

            if cskill is None:
                if req.is_required:
                    gaps.append(req.skill_name)
                else:
                    matched_score += req.weight * 0.3
            else:
                level_diff = cskill.proficiency_level - req.required_level
                if level_diff >= 0:
                    match = 1.0
                    if level_diff >= 2:
                        strengths.append(f"{req.skill_name} (+{level_diff} levels)")
                else:
                    match = max(0.3, 1.0 + level_diff * 0.2)
                    if level_diff <= -2:
                        gaps.append(f"{req.skill_name} (need +{-level_diff} levels)")

                if cskill.verified:
                    match = min(1.0, match * 1.1)

                matched_score += req.weight * match

        return round(matched_score / total_weight, 4), gaps, strengths

    def calculate_experience_match(self, candidate: CandidateProfile, job: JobProfile) -> float:
        if candidate.experience_years is None:
            return 0.5
        if job.experience_min is None and job.experience_max is None:
            return 0.8

        exp = candidate.experience_years
        if job.experience_min and job.experience_max:
            if job.experience_min <= exp <= job.experience_max:
                return 1.0
            elif exp < job.experience_min:
                return max(0.3, exp / job.experience_min)
            else:
                return max(0.5, 1.0 - (exp - job.experience_max) / 10)
        elif job.experience_min:
            return 1.0 if exp >= job.experience_min else max(0.3, exp / job.experience_min)
        else:
            return 1.0 if exp <= job.experience_max else max(0.5, 1.0 - (exp - job.experience_max) / 10)

    def calculate_ncf_score(self, candidate: CandidateProfile, job: JobProfile) -> float:
        # Simplified NCF: combines skill overlap with experience fit
        job_skills = set(s.skill_name.lower() for s in job.required_skills)
        cand_skills = set(s.skill_name.lower() for s in candidate.skills)

        if not job_skills:
            return 0.5

        overlap = len(job_skills & cand_skills) / len(job_skills)
        verified_bonus = sum(1 for s in candidate.skills if s.verified) / max(len(candidate.skills), 1) * 0.2

        return min(1.0, overlap + verified_bonus)

    def match_candidates(self, request: CandidateMatchRequest) -> CandidateMatchResponse:
        start = time.time()

        matches = []
        for candidate in request.candidates:
            skill_score, gaps, strengths = self.calculate_skill_match(candidate, request.job)
            exp_score = self.calculate_experience_match(candidate, request.job)
            ncf_score = self.calculate_ncf_score(candidate, request.job)

            overall = skill_score * 0.5 + exp_score * 0.2 + ncf_score * 0.3

            if overall >= request.min_score:
                matches.append(MatchScore(
                    candidate_id=candidate.candidate_id,
                    job_id=request.job.job_id,
                    overall_score=round(overall, 4),
                    skill_match_score=skill_score,
                    experience_match_score=round(exp_score, 4),
                    ncf_score=round(ncf_score, 4),
                    skill_gaps=gaps,
                    skill_strengths=strengths,
                ))

        matches.sort(key=lambda x: x.overall_score, reverse=True)
        top_matches = matches[:request.top_k]

        return CandidateMatchResponse(
            job_id=request.job.job_id,
            matches=top_matches,
            total_candidates=len(request.candidates),
            processing_time_ms=round((time.time() - start) * 1000, 2),
        )

ncf_service = NCFService()
print("NCF service defined!")

## 4. Load the Models

In [ ]:
# Load models (this will download them on first run)
print("Loading models... This may take a minute on first run.")
print()

codebert_service.load_model()
print()

bert_service.load_model()
print()

ncf_service.load_model()
print()

print("All models loaded!")

## 5. Create FastAPI App

In [ ]:
# Create FastAPI app
app = FastAPI(
    title="VeriHire ML Service",
    version="0.1.0",
    description="ML service for code evaluation, text analysis, and candidate matching",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Health endpoints
@app.get("/health", response_model=HealthResponse)
def health_check():
    return HealthResponse(
        status="healthy",
        version="0.1.0",
        timestamp=datetime.now(timezone.utc).isoformat(),
    )

@app.get("/health/models", response_model=HealthResponse)
def health_models():
    return HealthResponse(
        status="healthy",
        version="0.1.0",
        timestamp=datetime.now(timezone.utc).isoformat(),
        models=[
            ModelStatus(name="CodeBERT", loaded=codebert_service.loaded, device=codebert_service.device),
            ModelStatus(name="BERT", loaded=bert_service.loaded, device=bert_service.device),
            ModelStatus(name="NCF", loaded=ncf_service.loaded, device=ncf_service.device),
        ],
    )

# Code evaluation
@app.post("/api/v1/evaluate/code", response_model=CodeEvaluationResponse)
def evaluate_code(request: CodeEvaluationRequest):
    if not request.code.strip():
        raise HTTPException(status_code=400, detail="Code cannot be empty")
    return codebert_service.evaluate(request)

# Text evaluation
@app.post("/api/v1/evaluate/text", response_model=TextEvaluationResponse)
def evaluate_text(request: TextEvaluationRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    return bert_service.evaluate(request)

# Candidate matching
@app.post("/api/v1/match/candidates", response_model=CandidateMatchResponse)
def match_candidates(request: CandidateMatchRequest):
    if not request.candidates:
        raise HTTPException(status_code=400, detail="At least one candidate required")
    return ncf_service.match_candidates(request)

print("FastAPI app created!")
print("Endpoints:")
print("  GET  /health")
print("  GET  /health/models")
print("  POST /api/v1/evaluate/code")
print("  POST /api/v1/evaluate/text")
print("  POST /api/v1/match/candidates")

## 6. Start the Server with ngrok

This will start the server and give you a public URL to use.

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("=" * 60)
print("ML SERVICE IS RUNNING!")
print("=" * 60)
print()
print(f"Public URL: {public_url}")
print()
print("Add this to your .env file:")
print(f"ML_SERVICE_URL={public_url}")
print()
print("=" * 60)
print("Keep this notebook running! The URL will stop working if you close it.")
print("=" * 60)

# Run the server
uvicorn.run(app, host="0.0.0.0", port=8000)